# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [11]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [12]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Lottery and Sales Data (for reference when merging)

In [13]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [14]:
print(locations_df.shape)
print(locations_df.isna().sum())
locations_df[locations_df['tract_geoid'].isna()]

(16793, 13)
agtno                      0
busnm                      0
busaddr                    0
buscity                    0
buszip                     0
chain                      0
bustype                    0
longitude                  0
latitude                   0
geometry                   0
bg_geoid                   3
tract_geoid                3
competitors_within_1_mi    0
dtype: int64


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi
5734,56984,DANDY MINI MART #26,10 MAIN ST,WELLSBURG,14894,chain,CONV. STORE,-76.72950,42.00145,b'\x01\x01\x00\x00\x00\xa6\x9b\xc4 \xb0.S\xc0#J{\x83/\x00E@',NaN,NaN,0
6685,70764,THE MAHOPAC INN,927 S LAKE BLVD,MAHOPAC,10541,independent,RESTAURANT-HOTEL,-73.73627,41.37214,b'\x01\x01\x00\x00\x00\xa9\xc14\x0c\x1foR\xc0H\xc4\x94H\xa2\xafD@',NaN,NaN,7
14643,114270,CLIFF'S LOCAL MARKETS #520,3931 ONEIDA ST,WASHINGTON MILLS,13479,chain,CONV. STORE,-76.95030,38.96106,b'\x01\x01\x00\x00\x00\xe2X\x17\xb7\xd1<S\xc04\xbf\x9a\x03\x04{C@',NaN,NaN,0


In [15]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [16]:
print(sales_df.shape)
print(sales_df.isna().sum())
sales_df.sample(5)

(14351, 15)
agtno                     0
total_sales               0
first_date                0
last_date                 0
first_pos_date            0
last_pos_date             0
pos_sales_days            0
num_cal_days              0
num_active_days           0
days_before_first_sale    0
days_after_last_sale      0
avg_cal_daily_sales       0
avg_active_daily_sales    0
use_active_period         0
avg_daily_sales           0
dtype: int64


,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
4840,112920,251990.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,690.383562,690.383562,False,690.383562
7603,116419,15940.5,2025-11-28,2025-12-31,2025-11-28,2025-12-31,34.0,34,34.0,0.0,0.0,468.838235,468.838235,False,468.838235
7448,116224,1840.0,2025-11-17,2025-12-31,2025-11-17,2025-12-31,33.0,45,45.0,0.0,0.0,40.888889,40.888889,False,40.888889
1711,105901,304176.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,833.360274,833.360274,False,833.360274
10260,59795,824239.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,363.0,365,365.0,0.0,0.0,2258.190411,2258.190411,False,2258.190411


## Overture Maps Data

In [17]:
interim_overture_file_path = Path(".").resolve().parent / "data/interim/external/overture_filtered.parquet"

interim_overture_pq = pq.ParquetFile(interim_overture_file_path)
interim_overture_first_100_rows = next(interim_overture_pq.iter_batches(batch_size=5000))
interim_overture_df = pa.Table.from_batches([interim_overture_first_100_rows]).to_pandas()

interim_overture_df['highest_hierarchy'] = interim_overture_df['taxonomy_hierarchy'].str[0]
print(interim_overture_df['highest_hierarchy'].unique())
print(interim_overture_df['basic_category'].nunique())
interim_overture_df.sample(20)

['services_and_business' 'sports_and_recreation' 'education' 'health_care'
 'arts_and_entertainment' 'shopping' 'lifestyle_services'
 'travel_and_transportation' 'community_and_government'
 'cultural_and_historic' 'geographic_entities' 'food_and_drink' 'lodging'
 None]
201


,id,geometry,longitude,latitude,basic_category,taxonomy_primary,taxonomy_hierarchy,confidence,brand_name,first_seen,highest_hierarchy
1918,60e9fd76-96e2-40cd-bfa1-b1c3a933367c,b'\x01\x01\x00\x00\x00&\xe0\xd7H\x122R\xc0VE\xb8\xc9\xa8fD@',-72.782366,40.802026,complementary_and_alternative_medicine,acupuncture,"[health_care, outpatient_care_facility, complementary_and_alternative_medicine, acupuncture]",0.919912,NaN,2025-06-25.0,health_care
4679,7bda4bd8-62b3-42eb-ad34-9ccfc2046e53,b'\x01\x01\x00\x00\x00\x04kA\x94TvR\xc0\xa1\xd3\x98\xb5\xb9RE@',-73.848912,42.646292,reproductive_perinatal_and_womens_care,obstetrics_and_gynecology,"[health_care, outpatient_care_facility, reproductive_perinatal_and_womens_care, obstetrics_and_gynecology]",0.919912,NaN,2025-06-25.0,health_care
621,a106d93b-7188-4b75-aeca-92de6555d657,"b'\x01\x01\x00\x00\x00i\x13\xcc=I,R\xc0\xc6e\xa2)\xb7vD@'",-72.691970,40.927465,electronics_store,electronics_store,"[shopping, specialty_store, electronics_store]",0.990219,Best Buy,2025-06-25.0,shopping
823,81a076e5-1d79-40bc-9c00-8193d7cdab89,"b""\x01\x01\x00\x00\x004L\n\x7f\xc8'R\xc0gi\x9fu\xaasD@""",-72.621612,40.903640,fire_station,fire_station,"[community_and_government, public_safety_service, fire_station]",0.919912,NaN,2025-06-25.0,community_and_government
2537,d0cfea2a-0210-4a72-baae-0f556f953128,b'\x01\x01\x00\x00\x008\xdd.\x0bX<R\xc0\x12\xa5\x9c\xb3\xaexD@',-72.942874,40.942831,vision_or_eye_care_clinic,optometry,"[health_care, outpatient_care_facility, vision_or_eye_care_clinic, optometry]",0.999971,Visionworks Doctors of Optometry,2025-06-25.0,health_care
3522,59078265-d3c2-40c3-8fcf-b76c1e80e7ef,"b'\x01\x01\x00\x00\x00\xa8\x88\x99\x9e,wR\xc0\xa2\x03y\x13O\x03E@'",-73.862098,42.025851,hardware_home_and_garden_store,nursery_and_gardening_store,"[shopping, specialty_store, hardware_home_and_garden_store, nursery_and_gardening_store]",0.990219,Flower Power,2025-06-25.0,shopping
2373,f5ab2023-4f5f-43ee-8a45-44c8a976f1c0,b'\x01\x01\x00\x00\x00\x15X|\x1abBR\xc0\xae2\xcf![{D@',-73.037238,40.963719,vehicle_service,boat_service,"[travel_and_transportation, vehicle_service, boat_service]",0.990219,NaN,2025-06-25.0,travel_and_transportation
1199,9f716c67-6e94-4a3d-b648-e0b172ac975c,"b'\x01\x01\x00\x00\x00N\x9bq\x1a""#R\xc0\x98*b""\xefnD@'",-72.548957,40.866673,restaurant,mexican_restaurant,"[food_and_drink, restaurant, latin_american_restaurant, mexican_restaurant]",0.963602,NaN,2025-06-25.0,food_and_drink
2470,29dd33ad-e6ed-40e6-9340-0580c3271426,b'\x01\x01\x00\x00\x009\xb8t\xccy=R\xc0*\xca\xa5\xf1\x0byD@',-72.960559,40.945677,personal_or_beauty_service,hair_salon,"[lifestyle_services, personal_or_beauty_service, hair_salon]",0.919912,NaN,2025-06-25.0,lifestyle_services
943,94c18fd8-1b91-423f-a36e-9494729acd6f,b'\x01\x01\x00\x00\x00\x89\\p\x06?*R\xc0\x7f\xe5\xf7\xecouD@',-72.660097,40.917478,social_or_community_service,social_or_community_service,"[community_and_government, social_or_community_service]",0.919912,NaN,2025-06-25.0,community_and_government


In [18]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [19]:
overture_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,commercial_places_within_1_mi,community_places_within_1_mi
65,110085,US FOOD MART DELI & GROCERY IN,952 CRANE ST,SCHENECTADY,12303,0,DELI-GROCERY,-73.94798,42.79824,"b'\x01\x01\x00\x00\x00\xc6PN\xb4\xab|R\xc0\xf5-s\xba,fE@'",360930214002,36093021400,130,70
59,56547,FOOD PLUS III MARKET,87 CLEVELAND DR,CHEEKTOWAGA,14215,0,GROCERY STORE,-78.79605,42.94386,b'\x01\x01\x00\x00\x00\xc5\xfe\xb2{\xf2\xb2S\xc0T\x00\x8cg\xd0xE@',360290102023,36029010202,120,28
33,69716,SMOKERS CHOICE #36,2851 W HENRIETTA RD SUITE B,ROCHESTER,14623,258,NEWS-SMOKE,-77.63330,43.09462,b'\x01\x01\x00\x00\x00\xa3#\xb9\xfc\x87hS\xc0\x15\xc6\x16\x82\x1c\x8cE@',360550130062,36055013006,310,15
1,68523,YAHIR & SISTERS GROCERY CORP,165 MCLEAN AVE,YONKERS,10705,0,GROCERY STORE,-73.88996,40.91546,b'\x01\x01\x00\x00\x00\xe2\xaf\xc9\x1a\xf5xR\xc0\xe9e\x14\xcb-uD@',361190013013,36119001301,267,84
62,96365,RITE AID #4277,115-10 MERRICK BLVD,JAMAICA,11434,296,DRUG STORE,-73.77773,40.68961,b'\x01\x01\x00\x00\x00\x8f\xc7\x0cT\xc6qR\xc0N\x7f\xf6#EXD@',360810280001,36081028000,230,124


## LEHD Data

In [20]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [21]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,rac_c000,rac_ce01,rac_ce02,rac_ce03,rac_ce01_prop,rac_ce02_prop
35,88265,CIBAO GROCERY CORP,89-04 86TH ST,WOODHAVEN,11421,independent,DELI-GROCERY,-73.85700,40.68939,b'\x01\x01\x00\x00\x00\x02+\x87\x16\xd9vR\xc0@\x18x\xee=XD@',360810008001,36081000800,89,726.0,113.0,253.0,360.0,0.155647,0.348485
48,77604,GEORGIA DELI,272 GEORGIA ST,BUFFALO,14201,independent,GROCERY STORE,-78.87996,42.89121,b'\x01\x01\x00\x00\x00qr\xbfCQ\xb8S\xc0 \xefU+\x13rE@',360290071022,36029007102,31,325.0,75.0,99.0,151.0,0.230769,0.304615


In [22]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [23]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,wac_cns01,wac_cns02,wac_cns03,wac_cns04,wac_cns05,wac_cns06,wac_cns07,wac_cns08,wac_cns09,wac_cns10,wac_cns11,wac_cns12,wac_cns13,wac_cns14,wac_cns15,wac_cns16,wac_cns17,wac_cns18,wac_cns19,wac_cns20,wac_c000,wac_ce01,wac_ce02,wac_ce03,wac_ce01_prop,wac_ce02_prop
38,109288,MATURO FOOD MARKET CORP,853 CLARKSON AVENUE,BROOKLYN,11203,independent,GROCERY STORE,-73.92974,40.65669,b'\x01\x01\x00\x00\x00\x1dr3\xdc\x80{R\xc0\x1c\xce\xfcj\x0eTD@',360470868001,36047086800,126,0.0,0.0,0.0,0.0,0.0,0.0,18.0,1.0,0.0,28.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,72.0,5.0,0.0,125.0,33.0,59.0,33.0,0.264000,0.472000
2,85214,HONG KONG SUPERMARKET,37-11 MAIN STREET,FLUSHING,11354,independent,SUPERMARKETS,-73.83094,40.76119,b'\x01\x01\x00\x00\x00\x0c<\xf7\x1e.uR\xc0h\x05\x86\xacnaD@',360810871002,36081087100,103,0.0,0.0,0.0,91.0,99.0,139.0,828.0,79.0,81.0,712.0,228.0,905.0,28.0,172.0,216.0,2880.0,16.0,2148.0,418.0,0.0,9040.0,2684.0,3566.0,2790.0,0.296903,0.394469


## AADT Data

In [24]:
#aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

#aadt_pq = pq.ParquetFile(aadt_file_path)
#aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
#aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"
aadt_df = pd.read_parquet(str(aadt_file_path))

In [25]:
print(aadt_df['aadt_count'].isna().sum(), len(aadt_df))
aadt_df.sample(10)

9730 16793


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,bg_geoid,tract_geoid,competitors_within_1_mi,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,Rmp,dist_to_read_m
10145,47693,KINNEY DRUGS #065,2085 RT 5 & 20,SENECA FALLS,13148,chain,DRUG STORE,-76.82808,42.90507,360999504001,36099950400,10,NaN,NaN,NaN,NaN,NaN,None,NaN
15494,114034,ETERNAL DELI,115 MAIN ST,HEMPSTEAD,11550,independent,DELI-GROCERY,-73.62585,40.71060,360594068011,36059406801,49,NaN,NaN,NaN,NaN,NaN,None,NaN
9780,39984,AVATAR ENTERPRISE INC,530 ROUTE 211 EAST,MIDDLETOWN,10941,independent,CONV. STORE,-74.37490,41.46190,360710110004,36071011000,12,NaN,NaN,NaN,NaN,NaN,None,NaN
3363,112753,5 BORO MARKET CORP,3900 5TH AVE,BROOKLYN,11232,independent,CONV. STORE,-74.00399,40.65117,360470088001,36047008800,120,54257.0,4082.0,6TH AVE,16,2.0,None,1303.466134
11108,68330,SHERIDAN WHOLESALE DISTRIBUTOR,255 SHERIDAN BOULEVARD,INWOOD,11096,independent,BEVERAGE,-73.75138,40.61810,360594110004,36059411000,31,NaN,NaN,NaN,NaN,NaN,None,NaN
16368,116128,A & Z MARKET CORPORATION,1547 CLIFFORD AVE,ROCHESTER,14609,independent,DELI-GROCERY,-77.58056,43.17570,360550084002,36055008400,34,NaN,NaN,NaN,NaN,NaN,None,NaN
4314,112613,GRAYS DELI INC,31 S WHITE ST,POUGHKEEPSIE,12601,independent,CONV. STORE,-73.91675,41.69750,360272209013,36027220901,25,54809.0,2462.0,S WHITE ST,19,2.0,None,3.933302
6109,73956,RITE AID #763,4220 DELEWARE AVENUE,TONAWANDA,14150,chain,DRUG STORE,-78.87544,42.99928,360290082011,36029008201,12,55192.0,574.0,Pirson Pkwy,19,2.0,None,1502.131637
4510,74579,JS LUNCHEONETTE INC,301 COLUMBUS AVE,TUCKAHOE,10707,independent,DELI-GROCERY,-73.82031,40.95928,361190048011,36119004801,8,54631.0,2124.0,Thompson St,19,2.0,None,113.694710
13901,108293,WILLISTON PARK DAIRY,675 WILLIS AVE,WILLISTON PARK,11596,independent,CONV. STORE,-73.64693,40.76331,360593033011,36059303301,16,NaN,NaN,NaN,NaN,NaN,None,NaN


## EPA Data

In [26]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

epa_pq = pq.ParquetFile(epa_file_path)
epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

In [27]:
epa_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,OBJECTID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
92,92204,AA SUPER DELI CORP,124-32 ROCKAWAY BLVD,SOUTH OZONE PARK,11420,0,DELI-GROCERY,-73.81275,40.67535,b'\x01\x01\x00\x00\x00\xbct\x93\x18\x04tR\xc0C\xadi\xdeqVD@',360810840001,36081084000,156957,36,81,84000,1,15.866547,47.863204,3.953458,1.212394,0.052713,0.158138,0.184495,0.606197,1.634096,0.105426,0.0,30.422333,16.166667,36,081,084000,1
46,91274,KINGS NAILS EIGHT LLC,3622 WHITE PLAINS RD,BRONX,10467,0,SPECIALTY(NON FOOD),-73.86480,40.87986,"b'\x01\x01\x00\x00\x00,e\x19\xe2XwR\xc0\x7f\x87\xa2@\x9fpD@'",360050380003,36005038000,160802,36,5,38000,3,53.551564,162.224876,9.173185,1.074337,6.115456,0.743772,0.247924,0.743772,0.000000,0.247924,0.0,32.274792,14.500000,36,005,038000,3
14,93307,SPEEDWAY #07827,810 METROPOLITAN AVE,BROOKLYN,11211,46,CONV. STORE,-73.94166,40.71472,b'\x01\x01\x00\x00\x00\xe3\xfcM(D|R\xc0\x08\xe6\xe8\xf1{[D@',360470477002,36047047700,158854,36,47,47700,2,39.219661,85.993609,11.429283,1.135100,0.626262,0.039141,2.896462,0.313131,2.074493,4.344693,0.0,33.777783,15.666667,36,047,047700,2
13,88848,RAYAN CONVENIENCE INC,304 WEST 14TH ST,NEW YORK,10014,0,DELI-GROCERY,-74.00325,40.74006,b'\x01\x01\x00\x00\x00\x91\xed|?5\x80R\xc0\xf4\x89<I\xba^D@',360610083002,36061008300,162038,36,61,8300,2,45.695293,73.571187,772.762105,16.407963,512.881150,22.759432,112.826796,96.683478,9.086130,2.117156,0.0,36.879645,13.333333,36,061,008300,2
10,79722,KANG MEI QIN INC,5101 6TH AVE,BROOKLYN,11220,0,GROCERY STORE,-74.00878,40.64313,b'\x01\x01\x00\x00\x00\xf86\xfd\xd9\x8f\x80R\xc0\xca\x89v\x15RRD@',360470098003,36047009800,154671,36,47,9800,3,44.589706,187.791261,11.790547,2.250923,0.428747,0.643121,2.465296,2.036549,1.607802,2.358109,0.0,23.053079,11.333333,36,047,009800,3


## ACS Data

In [28]:
acs_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_acs.parquet"

acs_pq = pq.ParquetFile(acs_file_path)
acs_first_100_rows = next(acs_pq.iter_batches(batch_size=100))
acs_df = pa.Table.from_batches([acs_first_100_rows]).to_pandas()

In [29]:
acs_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,no_vehicle_households_proportion
93,107358,GURU SPECIALTY PHARMACY INC,118-04 ROCKAWAY BLVD,SOUTH OZONE PARK,11420,0,DRUG STORE,-73.81987,40.67627,b'\x01\x01\x00\x00\x00->\x05\xc0xtR\xc0\n\xa2\xee\x03\x90VD@',360810102001,36081010200,119271.0,406.0,1183.0,0.336433,0.514793,0.148774,0.136265,0.139073,0.023652,0.682164,0.028501,0.529557,0.470443,0.852217,0.059113,0.088670,0.091133
0,55194,BOWL-O-DROME,2712 PINE AVE,NIAGARA FALLS,14301,0,BOWLING CENTERS,-79.02878,43.09513,b'\x01\x01\x00\x00\x00\xd9\xb1\x11\x88\xd7\xc1S\xc0*oG8-\x8cE@',360630210002,36063021000,60533.0,387.0,591.0,0.076142,0.527919,0.395939,0.182825,0.000000,0.025997,0.436548,0.000000,0.803618,0.196382,0.364341,0.175711,0.459948,0.080103
16,99488,WEST HARLEM DELI CORP,2040 5TH AVE,NEW YORK,10035,0,DELI-GROCERY,-73.94184,40.80699,b'\x01\x01\x00\x00\x00#\x84G\x1bG|R\xc0z\x19\xc5rKgD@',360610198001,36061019800,108250.0,1029.0,2212.0,0.129747,0.622061,0.248192,0.108619,0.269933,0.224784,0.724508,0.053086,0.448008,0.551992,0.424684,0.153547,0.421769,0.729835
86,57294,VILLAGE LIQUOR STORE OF DEPEW,6346 TRANSIT RD,DEPEW,14043,0,LIQUOR,-78.69714,42.92828,b'\x01\x01\x00\x00\x00\xf1.\x17\xf1\x9d\xacS\xc0\xf4\xc3\x08\xe1\xd1vE@',360290097021,36029009702,78889.0,574.0,1258.0,0.158188,0.595390,0.246423,0.165019,0.083262,0.096996,0.724777,0.000000,0.764808,0.235192,0.735192,0.012195,0.252613,0.031359
36,102113,DAN'S SUPREME SUPERMARKETS INC,10095 MAIN ROAD,MATTITUCK,11952,256,SUPERMARKETS,-72.53848,40.98609,"b'\x01\x01\x00\x00\x005c\xd1tv""R\xc0\xd1tv28~D@'",361031700043,36103170004,149423.0,448.0,1190.0,0.259664,0.716807,0.023529,0.059041,0.342404,0.137188,0.884034,0.285171,0.794643,0.205357,0.660714,0.171875,0.167411,0.058036


## Master Data Sample

In [ ]:
master_file_path = Path(".").resolve().parent / "data/final/master_data.parquet"
master_df = pd.read_parquet(str(master_file_path))

print(master_df.shape)
print(master_df.columns)
print(master_df['Rmp'].describe())
master_df.sample(5)

(14341, 58)
Index(['agtno', 'avg_daily_sales', 'chain', 'bustype', 'longitude', 'latitude',
       'bg_geoid', 'tract_geoid', 'competitors_within_1_mi', 'aadt_count',
       'aadt_functional_class', 'aadt_lanes', 'Rmp', 'dist_to_read_m',
       'median_household_income', 'total_population', 'age_18_29_proportion',
       'age_30_64_proportion', 'poverty_rate',
       'bachelors_degree_proportion_25plus',
       'post_grad_degree_proportion_25plus', 'labor_force_participation_rate',
       'unemployment_rate', 'owner_occupied_proportion',
       'family_households_proportion', 'no_vehicle_households_proportion',
       'D1B', 'D1C', 'D3A', 'NatWalkInd', 'commercial_places_within_1_mi',
       'community_places_within_1_mi', 'rac_c000', 'rac_ce01_prop',
       'rac_ce02_prop', 'wac_cns01', 'wac_cns02', 'wac_cns03', 'wac_cns04',
       'wac_cns05', 'wac_cns06', 'wac_cns07', 'wac_cns08', 'wac_cns09',
       'wac_cns10', 'wac_cns11', 'wac_cns12', 'wac_cns13', 'wac_cns14',
       'wac_cns15'

,agtno,avg_daily_sales,chain,bustype,longitude,latitude,bg_geoid,tract_geoid,competitors_within_1_mi,aadt_count,aadt_functional_class,aadt_lanes,Rmp,dist_to_read_m,median_household_income,total_population,age_18_29_proportion,age_30_64_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,family_households_proportion,no_vehicle_households_proportion,D1B,D1C,D3A,NatWalkInd,commercial_places_within_1_mi,community_places_within_1_mi,rac_c000,rac_ce01_prop,rac_ce02_prop,wac_cns01,wac_cns02,wac_cns03,wac_cns04,wac_cns05,wac_cns06,wac_cns07,wac_cns08,wac_cns09,wac_cns10,wac_cns11,wac_cns12,wac_cns13,wac_cns14,wac_cns15,wac_cns16,wac_cns17,wac_cns18,wac_cns19,wac_cns20,wac_c000,wac_ce01_prop,wac_ce02_prop
8568,37283,491.831044,independent,RESTAURANT-HOTEL,-73.796770,42.677520,360010003021,36001000302,22.0,NaN,NaN,NaN,None,NaN,39307.0,1629.0,0.510129,0.304481,0.235186,0.299111,0.209378,0.677045,0.044184,0.270677,0.175439,0.251462,NaN,NaN,NaN,NaN,138.0,57.0,955.0,0.167539,0.233508,0.0,0.0,0.0,155.0,31.0,106.0,1111.0,543.0,290.0,2125.0,39.0,57.0,360.0,340.0,410.0,1537.0,97.0,262.0,159.0,122.0,7744.0,0.109504,0.181818
12266,77723,212.681818,independent,RESTAURANT-HOTEL,-77.282920,43.227260,361170201061,36117020106,4.0,NaN,NaN,NaN,None,NaN,49219.0,1696.0,0.191038,0.531840,0.075202,0.051195,0.081911,0.535673,0.048335,0.535782,0.451036,0.231638,NaN,NaN,NaN,NaN,36.0,10.0,1067.0,0.179007,0.268978,16.0,0.0,0.0,92.0,8.0,16.0,97.0,2.0,13.0,10.0,32.0,1.0,0.0,17.0,0.0,49.0,0.0,41.0,28.0,107.0,529.0,0.279773,0.294896
6788,115507,1225.394366,independent,CONV. STORE,-73.683380,40.677310,360594106001,36059410600,19.0,NaN,NaN,NaN,None,NaN,NaN,943.0,0.132556,0.371156,0.062893,0.174973,0.219512,0.455782,0.000000,0.672544,0.811083,0.032746,17.044013,0.526230,27.938357,14.666667,116.0,38.0,575.0,0.130435,0.219130,0.0,0.0,0.0,4.0,0.0,3.0,0.0,0.0,8.0,0.0,0.0,5.0,0.0,0.0,0.0,1.0,0.0,0.0,11.0,0.0,32.0,0.218750,0.375000
8702,38229,3216.598630,independent,DELI-GROCERY,-74.362473,41.542212,360710109023,36071010902,2.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,528.0,0.174242,0.191288,20.0,0.0,0.0,27.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,8.0,0.0,13.0,1.0,17.0,0.0,0.0,17.0,0.0,111.0,0.144144,0.423423
5876,114422,694.763014,independent,DELI-GROCERY,-73.812550,40.704890,360810214003,36081021400,99.0,NaN,NaN,NaN,None,NaN,88382.0,2202.0,0.146231,0.723433,0.252886,0.147675,0.259070,0.641689,0.231423,0.472356,0.712740,0.341346,59.704716,10.442858,34.030423,19.000000,680.0,253.0,894.0,0.165548,0.269575,0.0,0.0,0.0,19.0,0.0,0.0,345.0,6.0,1.0,11.0,8.0,7.0,0.0,1.0,121.0,17.0,0.0,518.0,49.0,0.0,1103.0,0.186763,0.425204


## Missingness and Variance Check

In [32]:
def summarize_numeric_features(df, numeric_cols):

    summary = pd.DataFrame({
        "pct_missing": df[numeric_cols].isna().mean() * 100,
        "pct_zero": (df[numeric_cols] == 0).mean() * 100,
        "pct_mode": [df[col].value_counts(normalize=True, dropna=True).max() * 100 for col in numeric_cols],
        "mode": [df[col].value_counts(dropna=True).idxmax() if df[col].notna().any() else pd.NA for col in numeric_cols],
        "mean": df[numeric_cols].mean(),
        "std": df[numeric_cols].std(),
        "iqr": df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),
        "cv": (df[numeric_cols].std()) / (df[numeric_cols].mean().abs()),
        "n_unique": df[numeric_cols].nunique()
    })

    return summary.sort_values("pct_missing", ascending=False)

def summarize_categorical_features(df, categorical_cols):
    results = {}

    for col in categorical_cols:

        pct_missing = df[col].isna().mean() * 100

        distribution = df[col].value_counts(dropna=False, normalize=True).mul(100).rename("pct").to_frame()

        results[col] = {"pct_missing": pct_missing, "distribution": distribution}

    return results

In [ ]:
numeric_features = ['competitors_within_1_mi', 'aadt_count', 'aadt_lanes', 'dist_to_read_m',
       'median_household_income', 'total_population', 'age_18_29_proportion', 'age_30_64_proportion',
       'poverty_rate', 'bachelors_degree_proportion_25plus',
       'post_grad_degree_proportion_25plus', 'labor_force_participation_rate',
       'unemployment_rate', 'owner_occupied_proportion',
       'family_households_proportion', 'no_vehicle_households_proportion',
       'D1B', 'D1C', 'D3A', 'NatWalkInd', 'commercial_places_within_1_mi',
       'community_places_within_1_mi', 'rac_c000', 'rac_ce01_prop',
       'rac_ce02_prop', 'wac_cns01', 'wac_cns02', 'wac_cns03', 'wac_cns04',
       'wac_cns05', 'wac_cns06', 'wac_cns07', 'wac_cns08', 'wac_cns09',
       'wac_cns10', 'wac_cns11', 'wac_cns12', 'wac_cns13', 'wac_cns14',
       'wac_cns15', 'wac_cns16', 'wac_cns17', 'wac_cns18', 'wac_cns19',
       'wac_cns20', 'wac_c000', 'wac_ce01_prop', 'wac_ce02_prop']

numeric_features = ['aadt_count', 'dist_to_read_m',
                    'median_household_income', 'households_total', 'total_population',
                    'age_18_29_proportion', 'age_30_64_proportion', 'age_65_plus_proportion',
                    'poverty_rate', 'bachelors_degree_proportion_25plus', 'post_grad_degree_proportion_25plus',
                    'labor_force_participation_rate', 'unemployment_rate',
                    'owner_occupied_proportion', 'renter_occupied_proportion',
                    'family_households_proportion', 'nonfamily_multiperson_households_proportion',
                    'living_alone_households_proportion', 'no_vehicle_households_proportion',
                    'D1A', 'D1B', 'D1C', 'D1C8_RET', 'D1C8_OFF', 'D1C8_IND', 'D1C8_SVC', 'D1C8_ENT',
                    'D1C8_ED', 'D1C8_HLTH', 'D1C8_PUB', 'D3A', 'NatWalkInd',
                    'gas_station_within_0.5_mi', 'gas_station_within_1_mi', 'arts_and_entertainment_within_0.5_mi',
                    'arts_and_entertainment_within_1_mi', 'shopping_within_0.5_mi',
                    'shopping_within_1_mi', 'social_or_community_service_within_0.5_mi',
                    'social_or_community_service_within_1_mi', 'restaurant_within_0.5_mi',
                    'restaurant_within_1_mi', 'bar_within_0.5_mi', 'bar_within_1_mi',
                    'rac_c000', 'rac_ce01', 'rac_ce02', 'rac_ce03', 'wac_c000', 'wac_cns07',
                    'wac_cns18', 'wac_ce01', 'wac_ce02', 'wac_ce03']

categorical_features = ['chain', 'bustype', 'aadt_functional_class', 'aadt_lanes']

numeric_summary = summarize_numeric_features(master_df, numeric_features)
categorical_summary = summarize_categorical_features(master_df, categorical_features)

numeric_summary

KeyError: "['households_total', 'age_65_plus_proportion', 'renter_occupied_proportion', 'nonfamily_multiperson_households_proportion', 'living_alone_households_proportion', 'D1A', 'D1C8_RET', 'D1C8_OFF', 'D1C8_IND', 'D1C8_SVC', 'D1C8_ENT', 'D1C8_ED', 'D1C8_HLTH', 'D1C8_PUB', 'gas_station_within_0.5_mi', 'gas_station_within_1_mi', 'arts_and_entertainment_within_0.5_mi', 'arts_and_entertainment_within_1_mi', 'shopping_within_0.5_mi', 'shopping_within_1_mi', 'social_or_community_service_within_0.5_mi', 'social_or_community_service_within_1_mi', 'restaurant_within_0.5_mi', 'restaurant_within_1_mi', 'bar_within_0.5_mi', 'bar_within_1_mi', 'rac_ce01', 'rac_ce02', 'rac_ce03', 'wac_ce01', 'wac_ce02', 'wac_ce03'] not in index"

In [54]:
categorical_summary['chain']

{'pct_missing': np.float64(0.0),
 'distribution':              pct
 chain           
 0      74.822927
 30      3.345039
 23      2.208202
 46      1.720136
 296     1.410630
 ...          ...
 56      0.005952
 336     0.005952
 337     0.005952
 303     0.005952
 103     0.005952
 
 [106 rows x 1 columns]}

## Correlation

In [38]:
correlation_file_path = Path(".").resolve().parent / "src/correlation_test/high_correlation_test.csv"
correlation_df = pd.read_csv(str(correlation_file_path))

correlation_df

,variable_1,variable_2,correlation,abs_correlation
0,owner_occupied_proportion,renter_occupied_proportion,-1.000,1.000
1,wac_c000,wac_ce03,0.974,0.974
2,arts_and_entertainment_within_1_mi,shopping_within_1_mi,0.971,0.971
3,shopping_within_1_mi,restaurant_within_1_mi,0.970,0.970
4,shopping_within_1_mi,social_or_community_service_within_1_mi,0.968,0.968
5,restaurant_within_0.5_mi,restaurant_within_1_mi,0.965,0.965
6,arts_and_entertainment_within_1_mi,social_or_community_service_within_1_mi,0.962,0.962
7,arts_and_entertainment_within_1_mi,restaurant_within_1_mi,0.955,0.955
8,restaurant_within_1_mi,bar_within_1_mi,0.953,0.953
9,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,0.951,0.951
